# Silver Employee Transformation

This notebook transforms bronze employee data into curated silver datasets.

* Load bronze Delta data for CSV, JSON, and XML sources
* Apply cleansing rules such as positive age filtering and duplicate removal
* Standardize null handling and derive full names
* Write transformed data into silver Delta locations
* Read back silver outputs for validation


## CSV Employee Transformation

This section loads bronze employee records from the CSV ingestion flow, applies silver-layer quality rules, and validates the silver output.


In [0]:
# Load the bronze employee Delta data generated from the CSV ingestion process.
bronze_df = spark.read.format("delta").load(
    "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/bronze/employee/"
)

display(bronze_df)

In [0]:
# Keep only employee records with a valid positive age for the silver layer.
silver_df = bronze_df.filter(bronze_df.Age > 0)

display(silver_df)

In [0]:
# Remove duplicate employee records based on the employee identifier.
silver_df = silver_df.dropDuplicates(["Employee_id"])

display(silver_df)

In [0]:
# Replace missing salary values with 0 to standardize downstream reporting.
silver_df = silver_df.fillna({"Salary": 0})

display(silver_df)

In [0]:
# Derive a full name column by combining the employee first and last names.
from pyspark.sql.functions import concat_ws

silver_df = silver_df.withColumn(
    "Full_Name",
    concat_ws(" ", silver_df.First_Name, silver_df.Last_Name)
)

display(silver_df)

In [0]:
# Write the transformed CSV-based employee dataset to the silver Delta location.
silver_df.write \
.mode("overwrite") \
.format("delta") \
.save("abfss://landing@bronzeadlsstorage.dfs.core.windows.net/silver/employee/")

In [0]:
# Read the silver CSV Delta data back to confirm the transformation output.
final_silver_df = spark.read.format("delta").load(
    "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/silver/employee/"
)

display(final_silver_df)

## JSON Employee Transformation

This section transforms bronze JSON employee records into a cleaned silver dataset with standardized name fields and validation output.


## XML Employee Transformation

This section transforms bronze XML employee records into a cleaned silver dataset with standardized name fields and validation output.


In [0]:
# Transform bronze JSON employee data into a silver dataset by filtering, deduplicating,
# standardizing salary values, flattening nested name fields, and validating the final output.
from pyspark.sql.functions import col, concat_ws

bronze_json_df = spark.read.format("delta").load(
    "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/bronze/EMPJSON/"
)

silver_json_df = bronze_json_df.filter(col("Age") > 0)
silver_json_df = silver_json_df.dropDuplicates(["Employee_id"])
silver_json_df = silver_json_df.fillna({"Salary": 0})
silver_json_df = silver_json_df.withColumn("First_Name", col("Customer.First_Name"))
silver_json_df = silver_json_df.withColumn("Last_Name", col("Customer.Last_Name"))
silver_json_df = silver_json_df.withColumn(
    "Full_Name",
    concat_ws(" ", col("First_Name"), col("Last_Name"))
).drop("Customer")

silver_json_df.write \
    .mode("overwrite") \
    .format("delta") \
    .save("abfss://landing@bronzeadlsstorage.dfs.core.windows.net/silver/employee_json/")

final_silver_json_df = spark.read.format("delta").load(
    "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/silver/employee_json/"
)

display(final_silver_json_df)

In [0]:
# Transform bronze XML employee data into a silver dataset by filtering, deduplicating,
# standardizing salary values, flattening nested name fields, and validating the final output.
from pyspark.sql.functions import col, concat_ws

bronze_xml_df = spark.read.format("delta").load(
    "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/bronze/EMPXML/"
)

silver_xml_df = bronze_xml_df.filter(col("Age") > 0)
silver_xml_df = silver_xml_df.dropDuplicates(["Employee_id"])
silver_xml_df = silver_xml_df.fillna({"Salary": 0})
silver_xml_df = silver_xml_df.withColumn("First_Name", col("Customer.First_Name"))
silver_xml_df = silver_xml_df.withColumn("Last_Name", col("Customer.Last_Name"))
silver_xml_df = silver_xml_df.withColumn(
    "Full_Name",
    concat_ws(" ", col("First_Name"), col("Last_Name"))
).drop("Customer")

silver_xml_df.write \
    .mode("overwrite") \
    .format("delta") \
    .save("abfss://landing@bronzeadlsstorage.dfs.core.windows.net/silver/employee_xml/")

final_silver_xml_df = spark.read.format("delta").load(
    "abfss://landing@bronzeadlsstorage.dfs.core.windows.net/silver/employee_xml/"
)

display(final_silver_xml_df)